# 01 - Data Exploration

This notebook explores the UCI Individual Household Electric Power Consumption dataset after preprocessing.

**Prerequisites:** Run `python -m src.data_download` and `python -m src.data_preprocessing` first.

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from src.data_preprocessing import load_hourly_data
from src.config import TARGET

%matplotlib inline
sns.set_style('whitegrid')

## 1. Load the Hourly Data

In [ ]:
df = load_hourly_data()
print(f'Shape: {df.shape}')
print(f'Date range: {df.index.min()} to {df.index.max()}')
df.head()

## 2. Summary Statistics

In [ ]:
df.describe().round(3)

## 3. Missing Values Check

In [ ]:
print('Missing values per column:')
print(df.isna().sum())
print(f'\nTotal missing: {df.isna().sum().sum()}')

## 4. Consumption Over Time

In [ ]:
fig, ax = plt.subplots(figsize=(16, 5))
ax.plot(df.index, df[TARGET], linewidth=0.3, color='steelblue', alpha=0.7)
ax.set_xlabel('Date')
ax.set_ylabel('Global Active Power (kW)')
ax.set_title('Hourly Electricity Consumption Over Time')
plt.tight_layout()
plt.show()

## 5. Daily Pattern (Average by Hour of Day)

In [ ]:
df_copy = df.copy()
df_copy['hour'] = df_copy.index.hour
daily_avg = df_copy.groupby('hour')[TARGET].mean()
daily_std = df_copy.groupby('hour')[TARGET].std()

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(daily_avg.index, daily_avg.values, marker='o', color='darkorange', linewidth=2, label='Mean')
ax.fill_between(daily_avg.index, daily_avg - daily_std, daily_avg + daily_std, alpha=0.2, color='darkorange', label='+/- 1 std')
ax.set_xlabel('Hour of Day')
ax.set_ylabel('Global Active Power (kW)')
ax.set_title('Average Daily Consumption Pattern')
ax.set_xticks(range(0, 24, 2))
ax.legend()
plt.tight_layout()
plt.show()

## 6. Weekly Pattern (Average by Day of Week)

In [ ]:
df_copy['dayofweek'] = df_copy.index.dayofweek
weekly_avg = df_copy.groupby('dayofweek')[TARGET].mean()
day_names = ['Mon', 'Tue', 'Wed', 'Thu', 'Fri', 'Sat', 'Sun']

fig, ax = plt.subplots(figsize=(8, 5))
ax.bar(day_names, weekly_avg.values, color='seagreen')
ax.set_xlabel('Day of Week')
ax.set_ylabel('Global Active Power (kW)')
ax.set_title('Average Weekly Consumption Pattern')
plt.tight_layout()
plt.show()

## 7. Monthly Pattern (Seasonal)

In [ ]:
df_copy['month'] = df_copy.index.month
monthly_avg = df_copy.groupby('month')[TARGET].mean()
month_names = ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec']

fig, ax = plt.subplots(figsize=(10, 5))
ax.bar(month_names, monthly_avg.values, color='coral')
ax.set_xlabel('Month')
ax.set_ylabel('Global Active Power (kW)')
ax.set_title('Average Monthly Consumption (Seasonal Pattern)')
plt.tight_layout()
plt.show()

## 8. Feature Correlation

In [ ]:
fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(df.corr(), annot=True, fmt='.2f', cmap='coolwarm', center=0, square=True, ax=ax, linewidths=0.5)
ax.set_title('Feature Correlation Heatmap')
plt.tight_layout()
plt.show()

## 9. Consumption Distribution

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
ax1.hist(df[TARGET], bins=100, edgecolor='black', color='steelblue', alpha=0.7)
ax1.set_xlabel('Global Active Power (kW)')
ax1.set_ylabel('Frequency')
ax1.set_title('Consumption Distribution')

df_copy.boxplot(column=TARGET, by='hour', ax=ax2, grid=False)
ax2.set_xlabel('Hour of Day')
ax2.set_ylabel('Global Active Power (kW)')
ax2.set_title('Consumption by Hour')
plt.suptitle('')
plt.tight_layout()
plt.show()

## 10. Key Observations

- **Daily pattern**: Consumption peaks in the evening (18:00-20:00) and is lowest during nighttime (02:00-06:00).
- **Weekly pattern**: Weekend consumption is slightly different from weekdays.
- **Seasonal pattern**: Higher consumption in winter months (heating) and lower in summer.
- **Correlation**: Global_active_power is highly correlated with Global_intensity (as expected, since P = V * I).
- **Distribution**: Right-skewed, with most values concentrated between 0.3 and 1.6 kW.
- **No missing values** after preprocessing.
- **34,589 hourly rows** spanning ~4 years.